In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [ ]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [ ]:
train_data.head(5)

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [ ]:
train_data.shape

(14732, 3)

In [ ]:
val_data.shape

(818, 3)

In [ ]:
train_data = train_data.sample(n=4000, random_state = 42).reset_index(drop = True)
val_data = val_data.sample(n = 500, random_state=42).reset_index(drop = True)

# Data Pre-Processing

In [ ]:
import re

def clean_data(text):

  text = re.sub(r"\r\n"," ",text)
  text = re.sub(r"\s+"," ",text)
  text = re.sub(r"<.*?>", " ", text)
  text = text.strip().lower()

  return text

In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

# Tokenize

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [ ]:
def tokenize(data):
  inputs = tokenizer(data["dialogue"], padding = "max_length", max_length= 512, truncation = True)
  targets = tokenizer(data["summary"], padding = "max_length", max_length= 150, truncation = True)

  inputs["labels"] = targets['input_ids']
  return inputs


In [ ]:
train_dataset = train_data.apply(tokenize, axis = 1).tolist()
val_dataset = val_data.apply(tokenize, axis = 1).tolist()

# Working with our model

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
import torch

if torch.backends.mps.is_available():
  device = torch.device("mps")
elif torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")
print("device: ", device)
model.to(device)

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy="epoch",
    save_strategy = "epoch",
    warmup_steps = 500
)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.595180,0.379835
2,0.396579,0.359477
3,0.373939,0.354506
4,0.361510,0.350525
5,0.355251,0.349052
6,0.350272,0.349237


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9054551035563151, metrics={'train_runtime': 1305.7738, 'train_samples_per_second': 18.38, 'train_steps_per_second': 2.297, 'total_flos': 3248203235328000.0, 'train_loss': 0.9054551035563151, 'epoch': 6.0})

In [ ]:
model.save_pretrained("./save_summary_model")
tokenizer.save_pretrained("./save_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./save_summary_model/tokenizer_config.json',
 './save_summary_model/tokenizer.json')

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("./save_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./save_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

# test the core logic for summarization

In [ ]:
def summarize_dialogue(dialogue):
  # clean data
  dialogue = clean_data(dialogue)

  # tokenize | inputs = dialogue tokens

  inputs = tokenizer(
      dialogue,
      padding = "max_length",
      max_length = 512,
      truncation = True,
      return_tensors = "pt"
  ).to(device)

  # generate dialogue summary: => token ids

  model.to(device)
  targets = model.generate(
  input_ids = inputs["input_ids"],
  attention_mask = inputs["attention_mask"],
  max_length = 150,
  num_beams = 4,             # => transformer will generate 4 different summaries and will give us the best among these
  early_stopping = True     # => as we get 4 summaries it will stop
  )


  # token ids convert to text => decoding

  summary = tokenizer.decode(targets[0], skip_special_tokens = True)    # => skip means skipping EOS,SEP,etc | we do not store them

  return summary

In [ ]:
test_dia = """
In the Joint Statement, Bhagat Singh and B.K. Dutt criticise Indian public leaders several times. A close reading of the text shows that this criticism is not because these leaders fail to resist imperialism, but because Bhagat Singh and Dutt disagree with the way these leaders resist it.

The text itself shows that public leaders were indeed resisting British rule through the Assembly. The statement says that "the national demand has been pressed by the people's representatives," that resolutions passed by the House were "trampled under foot," and that government proposals were "rejected as unacceptable by the elected members of the legislatures." This shows that the leaders were not silent or inactive. They used the tools available to them — debates, resolutions, and votes — to oppose government policy. This is important because it shows that Bhagat Singh and Dutt are not accusing these leaders of ignoring imperialism altogether. The leaders were clearly trying to resist it in their own way.

However, even though the leaders resisted, the text argues that their method of resistance achieved nothing. The statement describes the Assembly as a "hollow show and a mischievous make-believe," where resolutions end up in "the waste paper basket." It also says that public leaders "help the Government to squander public time and money on such a manifestly stage-managed exhibition of India's helpless subjection." Here, the word "help" is important. It suggests that by taking part in the Assembly, the leaders are not just failing to change anything — they are unintentionally supporting the appearance that Indians have a voice in governance, when the text argues this voice has no real power. So the criticism here is about the place and method chosen for resistance, not about whether resistance is happening.

This disagreement over method becomes clearer when the text directly responds to Dewan Chaman Lal, who is called a "pseudo-socialist" and who described the bombing as a "dastardly outrage," calling Bhagat Singh and Dutt "a disgrace to the country." The word "pseudo-socialist" suggests that Chaman Lal claims to share similar political goals as Bhagat Singh and Dutt, such as opposing exploitation, yet he strongly disapproves of their method of using a bomb as protest. This shows that the disagreement between them is not about whether to oppose the British government, but about how it should be opposed.

The essay also explains this disagreement using the idea of "violence" versus "Utopian non-violence." It says that force used aggressively is "violence" and is wrong, but force used for a "legitimate cause" has moral justification. It further states that the bombing marks "the end of an era of Utopian non-violence." This shows that the text sees two different approaches to resisting imperialism: one based on peaceful, constitutional methods (like the Assembly), and one based on revolutionary force. Bhagat Singh and Dutt believe the peaceful method is no longer effective, which is why they call it "Utopian," meaning unrealistic.

Finally, the text places Chaman Lal's words next to those of The Tribune of Lahore, a colonial newspaper that called them "lunatics." By putting an Indian leader's criticism side by side with a colonial newspaper's criticism, the text suggests something further: when an Indian leader condemns revolutionary resistance using the same language as the colonial press, that leader's opposition to British rule starts to look weaker, since he is repeating the coloniser's own judgment of what counts as acceptable protest. This shows that while the main disagreement is about method, the text also hints that defending only constitutional methods, in the same words the British use, can end up sounding like support for the imperial system rather than genuine resistance to it.

Overall, the Joint Statement criticises Indian public leaders not because they fail to resist imperialism, but because their method of resistance — working within the colonial Assembly — is shown in the text to be powerless and, in some cases, to end up echoing the coloniser's own language against revolutionary action.
"""
summary = summarize_dialogue(test_dia)
print("Summary = ",summary)

Summary =  bhagat singh and b.k. dutt criticise indian public leaders several times. bhagat singh and b.k. dutt disagree with the way they resist imperialism.
